# Causal inference from observational data

When treatment is not randomized, comparing treated and untreated units directly measures
association, not causation: the groups differ in ways that also drive the outcome. This notebook
works through the potential-outcomes framework on a simulated example where the bias is known,
then recovers the true effect three ways: regression adjustment, propensity-score inverse-probability
weighting, and a doubly robust combination. We close by asking how strong an unmeasured confounder
would have to be to explain away the result.


## Identification: the backdoor criterion

Causal effects are defined through interventions. The average treatment effect is
$\mathrm{ATE}=\mathbb E[Y(1)-Y(0)]$, where $Y(t)$ is the potential outcome under treatment $t$. The
problem is that we never see both potential outcomes for one unit.

Theorem (backdoor adjustment). If a set of covariates $X$ blocks every backdoor path from treatment $T$
to outcome $Y$ (no descendant of $T$ is in $X$, and conditioning on $X$ d-separates $T$ from $Y$ in the
graph with arrows out of $T$ removed), then the effect is identified by the adjustment formula
$$\mathbb E[Y(t)] = \mathbb E_X\big[\mathbb E[Y\mid T=t, X]\big].$$

Proof. Under the backdoor condition, $Y(t)\perp T\mid X$ (conditional ignorability). Then
$\mathbb E[Y\mid T=t,X]=\mathbb E[Y(t)\mid T=t,X]=\mathbb E[Y(t)\mid X]$, and averaging over $X$ gives
$\mathbb E_X\mathbb E[Y(t)\mid X]=\mathbb E[Y(t)]$. $\quad\blacksquare$

Regression adjustment, propensity-score weighting, and the doubly robust estimator in this notebook are
all estimators of this single formula.

Counterexample (do not condition on a collider). Adjusting for the wrong variable creates bias rather
than removing it. If $C$ is a common effect of $T$ and $Y$ (a collider), then $T$ and $Y$ are marginally
independent but become dependent given $C$: conditioning on a collider opens a spurious path and induces
an association where none existed. So "control for everything" is wrong; the graph dictates what to
adjust for and what to leave alone.

## 1. A confounded observational study

A job-training program is offered more often to motivated workers, and motivation also raises
later earnings on its own. Motivation is the confounder. We simulate this so the true average
treatment effect (ATE) is known, then show the naive treated-minus-untreated difference is biased.


In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.RandomState(0)
n = 4000
motivation = rng.normal(0, 1, n)                       # the confounder (also affects outcome)
p_treat = 1 / (1 + np.exp(-(0.4 + 1.2 * motivation)))  # motivated workers enroll more
T = (rng.random(n) < p_treat).astype(float)
TRUE_ATE = 2.0
Y = 5.0 + TRUE_ATE * T + 3.0 * motivation + rng.normal(0, 1, n)
naive = Y[T == 1].mean() - Y[T == 0].mean()
print('true ATE                 = %.2f' % TRUE_ATE)
print('naive difference in means= %.2f  (biased upward: motivated workers earn more anyway)' % naive)

true ATE                 = 2.00
naive difference in means= 4.77  (biased upward: motivated workers earn more anyway)


## 2. Regression adjustment (the backdoor criterion)

If we observe every common cause of treatment and outcome, conditioning on them blocks the
backdoor path and identifies the effect. Here motivation is the only confounder, so a regression
of outcome on treatment and motivation recovers the treatment coefficient as the ATE.


In [2]:
from mixle.ppl import Normal, Field, free
adj = Normal(free * Field('T') + free * Field('motivation') + free, free).fit(
    list(Y), given={'T': list(T.astype(float)), 'motivation': list(motivation)})
print('regression-adjusted ATE  = %.2f  (treatment coefficient, controlling for motivation)' %
      adj.result.coefficients['T']['mean'])

regression-adjusted ATE  = 2.03  (treatment coefficient, controlling for motivation)


## 3. Propensity scores and inverse-probability weighting

The propensity score is the probability of treatment given the covariates. Weighting each unit by
the inverse of its probability of the treatment it actually received creates a pseudo-population in
which treatment is independent of the covariates, so the weighted difference in means is unbiased.


In [3]:
from mixle.ppl import Bernoulli
ps_fit = Bernoulli(free * Field('motivation') + free).fit(list(T.astype(float)), given={'motivation': list(motivation)})
ps = np.asarray(ps_fit.result.predict({'motivation': list(motivation)}))   # propensity score P(T=1 | motivation)
w = T / ps + (1 - T) / (1 - ps)                        # inverse-propensity weights
ipw = (np.sum(w * T * Y) / np.sum(w * T)) - (np.sum(w * (1 - T) * Y) / np.sum(w * (1 - T)))
print('IPW estimate of the ATE  = %.2f' % ipw)

IPW estimate of the ATE  = 2.06


## 4. Did weighting balance the covariate?

A propensity model is only useful if the weights actually balance the confounder between groups.
The standardized mean difference (SMD) should be near zero after weighting; a common rule of thumb
flags imbalance above 0.1.


In [4]:
def smd(x, t, weights=None):
    if weights is None:
        weights = np.ones_like(t)
    m1 = np.average(x[t == 1], weights=weights[t == 1]); m0 = np.average(x[t == 0], weights=weights[t == 0])
    v1 = np.average((x[t == 1] - m1) ** 2, weights=weights[t == 1])
    v0 = np.average((x[t == 0] - m0) ** 2, weights=weights[t == 0])
    return (m1 - m0) / np.sqrt((v1 + v0) / 2)
print('SMD of motivation  before weighting = %.2f' % smd(motivation, T))
print('SMD of motivation  after  weighting = %.2f  (near 0 means balanced)' % smd(motivation, T, w))

SMD of motivation  before weighting = 1.05
SMD of motivation  after  weighting = 0.00  (near 0 means balanced)


## 5. Doubly robust estimation

The augmented IPW estimator combines the outcome regression and the propensity model and stays
consistent if either one is correct, not necessarily both. It is the safer default.


In [5]:
m1 = Normal(free * Field('motivation') + free, free).fit(list(Y[T == 1]), given={'motivation': list(motivation[T == 1])})
m0 = Normal(free * Field('motivation') + free, free).fit(list(Y[T == 0]), given={'motivation': list(motivation[T == 0])})
mu1 = np.asarray(m1.result.predict({'motivation': list(motivation)}))   # outcome model fit on the treated
mu0 = np.asarray(m0.result.predict({'motivation': list(motivation)}))   # outcome model fit on the controls
dr = np.mean(mu1 - mu0 + T * (Y - mu1) / ps - (1 - T) * (Y - mu0) / (1 - ps))
print('doubly robust ATE        = %.2f' % dr)

doubly robust ATE        = 2.06


## 6. Sensitivity to an unmeasured confounder

Every observational estimate rests on the untestable assumption of no unmeasured confounding. To see
what that assumption buys, we regenerate the data with a hidden variable U that drives both treatment
and outcome, then adjust only for the observed motivation. As U's influence grows, the estimate drifts
away from the truth, because the confounding through U is never controlled.


In [6]:
for gamma in [0.0, 0.5, 1.0, 2.0]:
    U = rng.normal(0, 1, n)                            # hidden: we never adjust for it
    pT = 1 / (1 + np.exp(-(0.4 + 1.2 * motivation + gamma * U)))
    Tg = (rng.random(n) < pT).astype(float)
    Yg = 5.0 + TRUE_ATE * Tg + 3.0 * motivation + gamma * U + rng.normal(0, 1, n)
    mg = Normal(free * Field('T') + free * Field('motivation') + free, free).fit(
        list(Yg), given={'T': list(Tg), 'motivation': list(motivation)})
    est = mg.result.coefficients['T']['mean']
    print('U strength gamma=%.1f  ->  motivation-adjusted estimate = %.2f  (true %.1f)' % (gamma, est, TRUE_ATE))
print('only gamma=0 is unbiased; a real unmeasured confounder pulls the estimate away from the truth.')

U strength gamma=0.0  ->  motivation-adjusted estimate = 2.01  (true 2.0)
U strength gamma=0.5  ->  motivation-adjusted estimate = 2.29  (true 2.0)
U strength gamma=1.0  ->  motivation-adjusted estimate = 2.89  (true 2.0)
U strength gamma=2.0  ->  motivation-adjusted estimate = 4.48  (true 2.0)
only gamma=0 is unbiased; a real unmeasured confounder pulls the estimate away from the truth.


## References

- Rubin, D. (1974). Estimating causal effects of treatments in randomized and nonrandomized studies. Journal of Educational Psychology.
- Rosenbaum, P. & Rubin, D. (1983). The central role of the propensity score in observational studies for causal effects. Biometrika.
- Robins, J., Rotnitzky, A. & Zhao, L. (1994). Estimation of regression coefficients when some regressors are not always observed. JASA (augmented IPW).
- Pearl, J. (2009). Causality: Models, Reasoning, and Inference. Cambridge University Press.
- VanderWeele, T. & Ding, P. (2017). Sensitivity analysis in observational research: introducing the E-value. Annals of Internal Medicine. https://doi.org/10.7326/M16-2607


## Exercises

1. Add a second confounder that affects treatment but not outcome (an instrument) and one that affects outcome but not treatment. Show which must be adjusted for and which must not, and what happens if you condition on a collider.
2. Replace the linear outcome model with a nonlinear one and compare regression adjustment, IPW, and the doubly robust estimator when the propensity model is misspecified but the outcome model is right, and vice versa.
3. Implement a bootstrap to put confidence intervals on the IPW and doubly robust estimates, and compare their variance.
4. Build an E-value sensitivity analysis: report the minimum strength of association an unmeasured confounder would need with both treatment and outcome to explain away the observed effect.
